## MISA (2024-2025)
- Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

- Izay misy hoe `YOUR CODE HERE` na `YOUR ANSWER HERE` ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
chatGPT, Claude AI

---

# Multinomial regression

In [25]:
from random import randrange
import numpy as np
from sklearn.metrics import mean_squared_error, log_loss
from sklearn.datasets import load_diabetes, load_iris, load_digits
from scipy.special import huber
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def grad_check_sparse(f, x, analytic_grad, num_checks=12, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [26]:
data = load_iris()
X_train2, y_train2 = data.data, data.target

W = np.random.randn(X_train2.shape[1], 3) * 0.0001

## Naive

In [27]:
def softmax_loss_naive(W, X, y, alpha):
    """
    Softmax loss function WITH FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    
    loss = 0.0
    dW = np.zeros_like(W)
    
    N = X.shape[0]
    C = W.shape[1]
    
    for i in range(N):
        scores = X[i] @ W  
        
        # Numerical stability: subtract max
        scores -= np.max(scores)
        
        # Compute softmax probabilities
        exp_scores = np.exp(scores)
        probs = exp_scores / np.sum(exp_scores)
        
        # Compute cross-entropy loss for this sample
        loss += -np.log(probs[y[i]])
        
        # Compute gradient
        probs[y[i]] -= 1
        dW += np.outer(X[i], probs)
    
    # Average over samples
    loss /= N
    dW /= N
    
    # Add L2 regularization
    loss += alpha * np.sum(W ** 2)
    dW += 2 * alpha * W
    
    return loss, dW

### Without regularization

In [28]:
loss, dW = softmax_loss_naive(W, X_train2, y_train2, 0.0)

f = lambda W: softmax_loss_naive(W, X_train2, y_train2, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: 0.028292 analytic: 0.028292, relative error: 6.453980e-10
numerical: 0.028292 analytic: 0.028292, relative error: 6.453980e-10
numerical: -0.169695 analytic: -0.169695, relative error: 2.528469e-10
numerical: -0.597418 analytic: -0.597418, relative error: 1.076067e-10
numerical: -0.043002 analytic: -0.043002, relative error: 2.977163e-10
numerical: -0.247267 analytic: -0.247267, relative error: 4.400420e-10
numerical: -0.169695 analytic: -0.169695, relative error: 2.528469e-10
numerical: -0.122384 analytic: -0.122384, relative error: 1.684255e-10
numerical: 0.318376 analytic: 0.318376, relative error: 6.436032e-11
numerical: 0.281534 analytic: 0.281534, relative error: 4.154391e-10
numerical: -0.122384 analytic: -0.122384, relative error: 1.684255e-10
numerical: -0.122384 analytic: -0.122384, relative error: 1.684255e-10


### With regularization

In [29]:
loss, dW = softmax_loss_naive(W, X_train2, y_train2, 2)

f = lambda W: softmax_loss_naive(W, X_train2, y_train2, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: 0.281751 analytic: 0.281751, relative error: 4.197272e-10
numerical: -0.034786 analytic: -0.034786, relative error: 3.784369e-09
numerical: 0.094192 analytic: 0.094192, relative error: 3.548201e-10
numerical: -0.043361 analytic: -0.043361, relative error: 3.283823e-10
numerical: -0.170893 analytic: -0.170893, relative error: 2.400361e-10
numerical: -0.043361 analytic: -0.043361, relative error: 3.283823e-10
numerical: 0.094192 analytic: 0.094192, relative error: 3.548201e-10
numerical: -0.246784 analytic: -0.246784, relative error: 4.377579e-10
numerical: 0.767592 analytic: 0.767592, relative error: 4.296480e-11
numerical: -0.598181 analytic: -0.598181, relative error: 1.034312e-10
numerical: -0.246784 analytic: -0.246784, relative error: 4.377579e-10
numerical: -0.598181 analytic: -0.598181, relative error: 1.034312e-10


## Vectorized

In [30]:
def softmax_loss_vectorized(W, X, y, alpha, fit_intercept=False):
    """
    Softmax loss function WITHOUT FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    loss = 0.0
    dW = np.zeros_like(W)
    
    N = X.shape[0]
    C = W.shape[1]
    
    # Compute scores for all samples at once
    scores = X @ W  
    
    scores -= np.max(scores, axis=1, keepdims=True)
    
    # Compute softmax probabilities
    exp_scores = np.exp(scores)  
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True) 
    
    # Compute cross-entropy loss
    correct_logprobs = -np.log(probs[np.arange(N), y])
    loss = np.mean(correct_logprobs)
    
    # Compute gradient
    dscores = probs.copy()  
    dscores[np.arange(N), y] -= 1 
    dW = X.T @ dscores  
    dW /= N
    
    # Add L2 regularization
    loss += alpha * np.sum(W ** 2)
    dW += 2 * alpha * W
    
    return loss, dW

### Without regularization

In [31]:
loss, dW = softmax_loss_vectorized(W, X_train2, y_train2, 0.0)

f = lambda W: softmax_loss_vectorized(W, X_train2, y_train2, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: -0.043002 analytic: -0.043002, relative error: 8.955464e-11
numerical: -0.043002 analytic: -0.043002, relative error: 8.955464e-11
numerical: -0.034267 analytic: -0.034267, relative error: 3.779233e-09
numerical: 0.028292 analytic: 0.028292, relative error: 8.416088e-10
numerical: -0.043002 analytic: -0.043002, relative error: 8.955464e-11
numerical: -0.597418 analytic: -0.597418, relative error: 8.902304e-11
numerical: 0.094093 analytic: 0.094093, relative error: 2.358358e-10
numerical: -0.597418 analytic: -0.597418, relative error: 8.902304e-11
numerical: -0.034267 analytic: -0.034267, relative error: 3.779233e-09
numerical: -0.034267 analytic: -0.034267, relative error: 3.779233e-09
numerical: 0.281534 analytic: 0.281534, relative error: 4.745912e-10
numerical: -0.043002 analytic: -0.043002, relative error: 8.955464e-11


### With regularization

In [32]:
loss, dW = softmax_loss_vectorized(W, X_train2, y_train2, 2)

f = lambda W: softmax_loss_vectorized(W, X_train2, y_train2, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: 0.767592 analytic: 0.767592, relative error: 6.466035e-11
numerical: -0.170893 analytic: -0.170893, relative error: 3.050022e-10
numerical: -0.043361 analytic: -0.043361, relative error: 5.567831e-11
numerical: 0.767592 analytic: 0.767592, relative error: 6.466035e-11
numerical: -0.122229 analytic: -0.122229, relative error: 1.053481e-10
numerical: -0.043361 analytic: -0.043361, relative error: 5.567831e-11
numerical: 0.281751 analytic: 0.281751, relative error: 4.788338e-10
numerical: 0.318556 analytic: 0.318556, relative error: 2.290546e-12
numerical: -0.170893 analytic: -0.170893, relative error: 3.050022e-10
numerical: -0.034786 analytic: -0.034786, relative error: 3.784369e-09
numerical: -0.598181 analytic: -0.598181, relative error: 8.487118e-11
numerical: 0.318556 analytic: 0.318556, relative error: 2.290546e-12


## Gradient descent

In [33]:
class LinearModel():
    def __init__(self, fit_intercept=True):
        self.W = None
        self.fit_intercept = fit_intercept

    def train(self, X, y, learning_rate=1e-3, alpha=0, num_iters=100, batch_size=200, verbose=False):
        if self.fit_intercept:
            # Add bias column of ones to X
            X = np.column_stack([X, np.ones(X.shape[0])])
            
        N, d = X.shape
        
        C = (np.max(y) + 1) 
        if self.W is None: # Initialization
            self.W = 0.001 * np.random.randn(d, C)

        # Run stochastic gradient descent to optimize W
        
        loss_history = []
        for it in range(num_iters):
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)
            # Hint: Use np.random.choice to generate indices
            indices = np.random.choice(N, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            
            # evaluate loss and gradient
            loss, dW = self.loss(X_batch, y_batch, alpha)
            loss_history.append(loss)

            # perform parameter update
            # Update the weights W using the gradient and the learning rate.
            self.W = self.W - learning_rate * dW
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
                
        return loss_history

    def predict(self, X):
        pass

    def loss(self, X_batch, y_batch, reg):
        pass

class MultinomialLogisticRegressor(LinearModel):
    """ Softmax regression """

    def loss(self, X_batch, y_batch, alpha):
        return softmax_loss_vectorized(self.W, X_batch, y_batch, alpha)
    
    def predict(self, X):
        """ 
        Inputs:
        - X: array of shape (N, D) 

        Returns:
        - y_pred: 1-dimensional array of length N, each element is an integer giving the predicted class 
        """
        if self.fit_intercept:
            X = np.column_stack([X, np.ones(X.shape[0])])
        
        # Compute scores
        scores = X @ self.W 
        
        # Get predictions as argmax of scores
        y_pred = np.argmax(scores, axis=1)
        
        return y_pred

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train2 = scaler.fit_transform(X_train2)

sk_model = LogisticRegression(fit_intercept=False)
sk_model.fit(X_train2, y_train2)
sk_pred = sk_model.predict(X_train2)
sk_accuracy = accuracy_score(y_train2, sk_pred)

model = MultinomialLogisticRegressor(fit_intercept=False)
model.train(X_train2, y_train2, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X_train2)
model_accuracy = accuracy_score(y_train2, pred)

print("Accuracy scikit-learn:", sk_accuracy)
print("Accuracy gradient descent model :", model_accuracy)
assert sk_accuracy - model_accuracy < 0.01

iteration 0 / 75000: loss 1.096848
iteration 10000 / 75000: loss 0.333086
iteration 20000 / 75000: loss 0.382554
iteration 30000 / 75000: loss 0.299104
iteration 40000 / 75000: loss 0.264484
iteration 50000 / 75000: loss 0.246435
iteration 60000 / 75000: loss 0.286687
iteration 70000 / 75000: loss 0.275986
Accuracy scikit-learn: 0.86
Accuracy gradient descent model : 0.8666666666666667


In [24]:
sk_model = LogisticRegression(fit_intercept=True)
sk_model.fit(X_train2, y_train2)
sk_pred = sk_model.predict(X_train2)
sk_accuracy = accuracy_score(y_train2, sk_pred)

model = MultinomialLogisticRegressor(fit_intercept=True)
model.train(X_train2, y_train2, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X_train2)
model_accuracy = accuracy_score(y_train2, pred)

print("Accuracy scikit-learn:", sk_accuracy)
print("Accuracy gradient descent model :", model_accuracy)
assert sk_accuracy - model_accuracy < 0.02

iteration 0 / 75000: loss 1.097564
iteration 10000 / 75000: loss 0.322028
iteration 20000 / 75000: loss 0.226077
iteration 30000 / 75000: loss 0.241000
iteration 40000 / 75000: loss 0.166312
iteration 50000 / 75000: loss 0.160061
iteration 60000 / 75000: loss 0.179132
iteration 70000 / 75000: loss 0.161333
Accuracy scikit-learn: 0.9733333333333334
Accuracy gradient descent model : 0.96
